# 03 — Restoration time with survival analysis (Objective 2)

**Rows:** (storm episode, county) units with an outage. **Duration:** hours from outage start until 90% of peak customers are restored. **Censored:** not restored within 14 days or before the data ends.

Prediction is framed *at the outage peak*, so peak size is a feature. SVI is left out here on purpose and studied in notebook 04.
Models: **Cox proportional hazards** (lifelines) and **random survival forest** (scikit-survival). Metrics: concordance index (C-index) and integrated Brier score on 2021–2022.

In [ ]:
# Works on Colab (Drive) or locally (repo checkout)
import os, sys, pathlib, warnings
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = pathlib.Path("/content/drive/MyDrive/LightsOut")
    ZIP = pathlib.Path("/content/drive/MyDrive/LightsOut_code.zip")
    if ZIP.exists():  # refresh code (src/, notebooks/) from the latest uploaded zip; data/ is untouched
        import zipfile; zipfile.ZipFile(ZIP).extractall(PROJECT)
    !pip -q install polars lifelines scikit-survival shap geopandas lightgbm
except ImportError:
    PROJECT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))
os.chdir(PROJECT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from lightsout import config as C, viz
viz.setup()
df = pd.read_parquet(C.PROCESSED / "model_table.parquet")
print(f"{len(df):,} (storm episode, county) units, {df.fips.nunique():,} counties, years {df.year.min()}-{df.year.max()}")

In [ ]:
from lightsout import models_survival as V
fit = V.fit_all(df)
for k, d in fit["data"].items():
    print(f"{k:5s} n={len(d):,}  censored={1 - d.restored.mean():.1%}  median hours={d.restore_hours.median():.1f}")

In [ ]:
res = V.results_table(fit, "test")
res.to_csv(C.REPORTS / "survival_results_test.csv")
(C.REPORTS / "survival_results_test.md").write_text(res.to_markdown())
res

## Cox model: hazard ratios

HR > 1 → restored **faster**; HR < 1 → slower. Numeric features are standardized (per 1 SD).

In [ ]:
cph = fit["cph"]
hr = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
hr = hr.reindex(hr["exp(coef)"].sub(1).abs().sort_values(ascending=False).index).head(15)
fig, ax = plt.subplots(figsize=(8, 5.5))
y = np.arange(len(hr))[::-1]
ax.errorbar(hr["exp(coef)"], y, xerr=[hr["exp(coef)"] - hr["exp(coef) lower 95%"],
            hr["exp(coef) upper 95%"] - hr["exp(coef)"]], fmt="o", color=viz.CATEGORICAL[0],
            ecolor=viz.MUTED, markersize=6, capsize=0)
ax.axvline(1, color=viz.AXIS, linewidth=1); ax.set_yticks(y, hr.index); ax.set_xscale("log")
ax.set_xlabel("Hazard ratio of restoration (log scale)"); ax.grid(axis="x")
ax.set_title("Cox PH: largest effects on restoration speed")
viz.save(fig, "09_cox_hazard_ratios"); fig

## Proportional-hazards check

If the assumption is violated for key covariates, the fix is to stratify on them (e.g. `strata=['storm_type']`) — the RSF does not need the assumption.

In [ ]:
from lifelines.statistics import proportional_hazard_test
tr = fit["data"]["train"]; Xtr = fit["X"]["train"]
# must use the same rows the model was fit on (lifelines reuses the fitted residuals)
ph = proportional_hazard_test(cph, pd.concat([Xtr, tr[[V.DURATION, V.EVENT]]], axis=1), time_transform="rank")
ph.summary.sort_values("p").head(10).round(4)

## Case studies: predicted vs observed restoration

In [ ]:
te, Xte = fit["data"]["test"], fit["X"]["test"]
picks = {"Harris Co., TX (Feb 2021)": (te.fips == 48201) & te.begin.between("2021-02-10", "2021-02-25"),
         "Orleans Par., LA (Ida)": (te.fips == 22071) & te.begin.between("2021-08-28", "2021-09-15"),
         "Jefferson Par., LA (Ida)": (te.fips == 22051) & te.begin.between("2021-08-28", "2021-09-15")}
times = np.linspace(0.25, 24 * 10, 200)
fig, ax = plt.subplots(figsize=(9, 4.5))
for (name, m), col in zip(picks.items(), viz.CATEGORICAL):
    if not m.any():
        print("not in test set:", name); continue
    i = te[m].sort_values("peak", ascending=False).index[0]
    fn = fit["rsf"].predict_survival_function(Xte.loc[[i]].to_numpy())[0]
    ax.plot(times, fn(np.clip(times, fn.x.min(), fn.x.max())), color=col, label=f"{name}: RSF prediction")
    ax.axvline(te.loc[i, "restore_hours"], color=col, linestyle="--", linewidth=1)
ax.set_xlabel("Hours since outage start (dashed = observed 90% restoration)")
ax.set_ylabel("P(not yet 90% restored)"); ax.legend(); ax.set_title("Predicted restoration curves, case studies")
viz.save(fig, "10_case_study_survival"); fig